<a href="https://colab.research.google.com/github/musab855/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ---- Setup -----------------------------------------------------------------
# Runs from the repo root (Colab badge clone or local clone). The token is read
# from Colab Secrets (same as Week 4), the environment, or a local .env file and
# is NEVER printed, stored, or written anywhere.
import os, sys, getpass, subprocess, importlib.util
from pathlib import Path

def find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "work").is_dir() and (p / "skills").is_dir():
            return p
    return None

root = find_repo_root(Path.cwd())
if root is None:
    # Fresh Colab badge runtime: it only downloaded this one .ipynb, so clone
    # the repo first (same pattern as Week 1 and Week 2).
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/musab855/flyrank-ml-internship.git"])
    root = Path.cwd() / "flyrank-ml-internship"
os.chdir(root)

missing = [p for p in ("duckdb", "huggingface_hub") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    importlib.invalidate_caches()

def load_token():
    # Colab Secrets first (Week 4 stored it as HF_TOKEN2, then HF_TOKEN).
    try:
        from google.colab import userdata
        for key in ("HF_TOKEN2", "HF_TOKEN"):
            try:
                token = userdata.get(key)
                if token:
                    return token
            except Exception:
                pass
    except Exception:
        pass
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    env_file = Path(".env")
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN"):
                return line.split("=", 1)[1].strip().strip(chr(34) + chr(39))
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

print("working directory:", Path.cwd())

working directory: C:\Users\Asus\Downloads\flyrank-ml-internship


In [2]:
# ---- Config ----------------------------------------------------------------
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 42          # every random step in this notebook uses this one seed
N_FOLDS = 5                # client-disjoint folds
BASE_FEATURES = ["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"]
IMPRESSIONS_THRESHOLD = 500   # the Week-4 rule's cutoff, unchanged
POSITION_THRESHOLD = 11       # the Week-4 rule's cutoff, unchanged

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"random_state = {RANDOM_STATE} — fixed, so a rerun reproduces every table below")

numpy 1.26.4 | pandas 2.1.1 | scikit-learn 1.3.1
random_state = 42 — fixed, so a rerun reproduces every table below


In [3]:
# ---- Data: same SQL, same window, same filters as Week 4 --------------------
# March 2026 only. Features come from days 1-15; the label is defined on days 16-31.
# The heavy scan runs once and is cached (the cache file is gitignored, never committed).
CACHE = Path("work/outputs/feature_frame_march.csv")

if CACHE.exists():
    df = pd.read_csv(CACHE)
    print(f"Cache hit: {len(df):,} rows from {CACHE}")
else:
    HF_TOKEN = load_token()
    import duckdb
    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"

    df = con.sql(f"""
    WITH daily_averages AS (
      SELECT
        content_hash_id,
        client_hash_id,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_1to15,
        AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) as gsc_impressions_16to31,
        AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) as gsc_clicks_16to31,
        AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) as position_16to31
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = TRUE
      GROUP BY content_hash_id, client_hash_id
    )
    SELECT
      content_hash_id AS content_id,
      client_hash_id,
      gsc_impressions_1to15 as gsc_impressions_avg,
      gsc_clicks_1to15 as gsc_clicks_avg,
      position_1to15 as position_avg,
      (CASE
         WHEN gsc_impressions_16to31 IS NULL OR gsc_impressions_1to15 IS NULL THEN NULL
         WHEN gsc_impressions_16to31 < gsc_impressions_1to15 THEN 1
         ELSE 0
       END) as is_declining_label
    FROM daily_averages
    """).df()

    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False)
    print(f"Scanned the warehouse (March 2026): {len(df):,} rows -> cached at {CACHE}")

Cache hit: 176,738 rows from work\outputs\feature_frame_march.csv


In [4]:
# Same filters as Week 4: undefined label or no position data -> dropped, not guessed.
n_before = len(df)
pos_nan  = df["position_avg"].isna()
pos_zero = df["position_avg"] == 0
lab_nan  = df["is_declining_label"].isna()

n_pos_nan, n_pos_zero = int(pos_nan.sum()), int(pos_zero.sum())
n_lab_nan_extra = int((lab_nan & ~pos_nan & ~pos_zero).sum())

print(f"Rows before filter: {n_before:,}")
print(f"Excluding position NaN (no GSC data days 1-15): {n_pos_nan:,}")
print(f"Excluding position == 0 (the 'no data' placeholder): {n_pos_zero:,}")
print(f"Excluding undefined labels (no GSC data in one half): {n_lab_nan_extra:,}")
print("Choice: undefined-label pages are dropped, not labelled declining or not declining.")

df = df[~pos_nan & ~pos_zero & ~lab_nan].copy()
df["is_declining_label"] = df["is_declining_label"].astype(int)
df = df.reset_index(drop=True)
# Canonical row order (content_id is unique): identical frames whether they
# came from the cache or from a fresh warehouse scan, so stable sorts,
# tied queues and the forest's bootstrap reproduce in any environment.
df = df.sort_values("content_id", kind="stable").reset_index(drop=True)
n_dropped = n_before - len(df)
assert n_dropped == n_pos_nan + n_pos_zero + n_lab_nan_extra

df["ctr"] = np.where(df["gsc_impressions_avg"] > 0,
                     df["gsc_clicks_avg"] / df["gsc_impressions_avg"], 0.0)

print(f"\nModeling frame: {len(df):,} pages, {df['client_hash_id'].nunique()} clients")
print(f"Base rate (share declining): {df['is_declining_label'].mean():.3f}")

Rows before filter: 176,738
Excluding position NaN (no GSC data days 1-15): 24,757
Excluding position == 0 (the 'no data' placeholder): 1,306
Excluding undefined labels (no GSC data in one half): 10,076
Choice: undefined-label pages are dropped, not labelled declining or not declining.



Modeling frame: 140,599 pages, 43 clients
Base rate (share declining): 0.465


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane is a **ranking** problem: score every page, order the list, a reviewer reads the top.
So I need models that output a *probability*, not a hard yes/no — Precision@K is computed on scores.

**Logistic Regression (the readable check).** Three or four inputs, one coefficient each. If the
directions are sensible (more impressions → more chance of a drop? worse position → ?), I can say
what the model learned in one sentence. It is also the ceiling for "is there a simple linear
signal here at all". Impressions and clicks pass through `log1p` first (they run 0.3 → 100,000/day), so one outlier cannot hijack a coefficient.

**Random Forest (the primary model).** Same inputs, but it can cut the space into regions —
"high impressions *and* deep position" behaves differently from either alone — and it does not
care that impressions runs from 0.3 to 100,000/day. It is the model I would actually deploy for
scoring, so it is the one that earns the headline comparison.

**What I deliberately did not run:** a gradient-boosting stack, tuning grids, or five more
classifiers. Complexity has to earn its keep against the rule, and two models — one simple, one
already strong — answer that question. Adding models until one wins is how a comparison stops
meaning anything.

**Stated before I look at any result:** the label is *"impressions fell in days 16–31"*.
`gsc_impressions_avg` (days 1–15) is measured long before the label window, so it is **not
leakage** — but it is close kin of the target, so I expect it to dominate whatever importance
measure I use. If anything looks *perfectly* predictive, that is a bug, not a win.

In [5]:
# The two models, exactly as configured. Fixed seeds; nothing is tuned after the fact.
def build_models():
    return {
        "logistic_regression": Pipeline([
            ("log", ColumnTransformer([("log", FunctionTransformer(np.log1p), [0, 1])],
                                      remainder="passthrough")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(class_weight="balanced", max_iter=1000,
                                         random_state=RANDOM_STATE)),
        ]),
        "random_forest": RandomForestClassifier(
            n_estimators=200, max_depth=10, min_samples_leaf=25,
            class_weight="balanced_subsample", n_jobs=-1,
            random_state=RANDOM_STATE),
    }

for name, model in build_models().items():
    print(f"{name}:")
    print("   ", model)

logistic_regression:
    Pipeline(steps=[('log',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('log',
                                                  FunctionTransformer(func=<ufunc 'log1p'>),
                                                  [0, 1])])),
                ('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])
random_forest:
    RandomForestClassifier(class_weight='balanced_subsample', max_depth=10,
                       min_samples_leaf=25, n_estimators=200, n_jobs=-1,
                       random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Client-grouped, 5 folds, every page scored exactly once out-of-fold.**
Pages from one client share seasonality, templates, and measurement quirks — if the model sees a
client's pages in training it can memorise *that client* instead of learning decline. So each
client is assigned to exactly one test fold; training and test clients never overlap. Labels and
features keep their Week-3/Week-4 windows (features = days 1–15 of March, label = days 16–31,
no overlap), and June — the last month in the panel — is never touched.

**Why folds instead of one 20% client holdout.** I tried the simple version first and the
client sizes wreck it — the cell below prints both problems: a single client holds **25,264 of
these 140,599 pages** (18.0% of the frame), and **802 of the 1,074 pages my rule flags** (74.7%)
come from that one client. Simulated random 20% client holdouts handed test sets anywhere from
**7,874 to 36,166 pages** with anywhere from **0 to 970 flagged pages** — one trial flagged
*nothing at all*, so Precision@50 would not even exist. Precision@50 then measures *which
clients got held out*, not which method is better. The folds below are shuffled once (seed 42)
and then filled to equalise **page counts**, and the notebook reports both the pooled
out-of-fold number (one honest figure over all 140,599 pages) and the per-fold spread.

**One honest asymmetry, stated up front:** the Week-4 rule has no training step — it is the same
function on any page — so its in-sample and out-of-fold numbers are identical by definition.
The two models are only ever scored on clients they were not trained on.

In [6]:
# ---- Why not a single 20% client holdout: the concentration, in numbers ------
sizes = df.groupby("client_hash_id").size().sort_values(ascending=False)
flagged_mask = ((df["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) &
                (df["position_avg"] >= POSITION_THRESHOLD))
flag_by_client = df.loc[flagged_mask, "client_hash_id"].value_counts()

print(f"Clients: {len(sizes):,}   pages: {len(df):,}")
print(f"Client size min / median / max: {sizes.min():,} / {int(sizes.median()):,} / {sizes.max():,}")
print(f"Largest client: {sizes.iloc[0]:,} pages ({sizes.iloc[0] / len(df):.1%} of the frame)")
print(f"Rule flags {int(flagged_mask.sum()):,} pages; the top client owns "
      f"{int(flag_by_client.iloc[0]):,} of them ({flag_by_client.iloc[0] / flagged_mask.sum():.1%})")

# What a random 20% client holdout would have handed the test set (5 trials, seed 42)
rng = np.random.default_rng(RANDOM_STATE)
clients = sizes.index.to_numpy()
print("\nRandom 20% client holdout, 5 trials:")
print(f"{'trial':>5} {'test pages':>12} {'flagged in test':>16}")
for t in range(5):
    holdout = rng.choice(clients, size=max(1, len(clients) // 5), replace=False)
    pages = int(sizes.loc[holdout].sum())
    fl = int(flag_by_client.reindex(holdout).fillna(0).sum())
    print(f"{t:>5} {pages:>12,} {fl:>16}")


Clients: 43   pages: 140,599
Client size min / median / max: 2 / 973 / 25,264
Largest client: 25,264 pages (18.0% of the frame)
Rule flags 1,074 pages; the top client owns 802 of them (74.7%)

Random 20% client holdout, 5 trials:
trial   test pages  flagged in test
    0       14,637              802
    1       18,804               27
    2       36,166              970
    3        9,243              167
    4        7,874                0


In [7]:
# Shuffled once, then packed into 5 folds of roughly equal PAGE count.
# Every client lands in exactly one test fold; no client appears in two.
def make_fold_assignment(frame, n_splits=N_FOLDS, seed=RANDOM_STATE):
    sizes = frame.groupby("client_hash_id").size()
    order = list(sizes.index)
    np.random.default_rng(seed).shuffle(order)      # shuffle so ties are not alphabetical
    fold_of_client, load = {}, [0] * n_splits
    for client in order:
        i = int(np.argmin(load))                    # always fill the lightest fold
        fold_of_client[client] = i
        load[i] += int(sizes[client])
    return frame["client_hash_id"].map(fold_of_client).astype(int)

df["fold"] = make_fold_assignment(df)

assert df["fold"].notna().all()
assert df.groupby("client_hash_id")["fold"].nunique().eq(1).all(), "a client spans two folds"

print(f"{'fold':>4} {'pages':>8} {'clients':>8} {'flagged':>8} {'base rate':>10}")
for f in range(N_FOLDS):
    part = df[df["fold"] == f]
    flagged = ((part["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) &
               (part["position_avg"] >= POSITION_THRESHOLD)).sum()
    assert part["is_declining_label"].nunique() == 2, f"fold {f} has one class only"
    print(f"{f:>4} {len(part):>8,} {part['client_hash_id'].nunique():>8} "
          f"{flagged:>8} {part['is_declining_label'].mean():>10.3f}")

# no page is left out: fold sizes must add back up to the full frame
assert int(sum((df["fold"] == f).sum() for f in range(N_FOLDS))) == len(df)
print("\nCheck: client-disjoint folds, both classes present, every page in exactly one test fold.")

fold    pages  clients  flagged  base rate
   0   32,790        8      171      0.487
   1   25,826        9       38      0.439
   2   26,662        7       35      0.547


   3   32,230       12       28      0.353
   4   23,091        7      802      0.524

Check: client-disjoint folds, both classes present, every page in exactly one test fold.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Same universe for everyone.** All 140,599 pages, scored three ways:
`rule` (my Week-4 `HIGH_VOLUME_POOR_POSITION`, ranked by impressions; pages it does not flag get
score 0 and stay out of its queue), `logistic_regression`, `random_forest`. Metrics: the ones the
assignment asks for (**Precision@10 / 20 / 50**) plus **ROC-AUC** and **average precision**, which
describe the *whole* list rather than only the first 50 rows, plus the **base rate** — the number
every method has to beat.

Model rows are out-of-fold (each page scored by a model that never saw its client). The rule row
is computed on the same pages. Two fairness references — `rule_full` (the flag, but every page
gets an impressions rank) and `impressions_only` (no flag at all) — join the table in the
fairness cell right after it, because the raw rule's whole-list AUC is ties by construction.


In [8]:
# ---- Metric helpers ---------------------------------------------------------
def rule_score(frame):
    """My Week-4 rule: flagged pages ranked by impressions, everything else 0."""
    flagged = (frame["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) & \
              (frame["position_avg"] >= POSITION_THRESHOLD)
    return np.where(flagged, frame["gsc_impressions_avg"], 0.0)

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")[:k]
    return float(np.asarray(labels)[order].mean())

def metric_bundle(labels, scores):
    return {
        "precision_at_10": precision_at_k(labels, scores, 10),
        "precision_at_20": precision_at_k(labels, scores, 20),
        "precision_at_50": precision_at_k(labels, scores, 50),
        "roc_auc": roc_auc_score(labels, scores),
        "avg_precision": average_precision_score(labels, scores),
    }

print("Metrics: precision_at_10 / precision_at_20 / precision_at_50 / roc_auc / avg_precision")
print(f"Base rate on the full frame: {df['is_declining_label'].mean():.3f}")

Metrics: precision_at_10 / precision_at_20 / precision_at_50 / roc_auc / avg_precision
Base rate on the full frame: 0.465


In [9]:
# ---- Out-of-fold training: 5 client folds x 2 models ------------------------
def run_oof(feature_cols):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = df["is_declining_label"].to_numpy()
    scores = {"rule": rule_score(df),
              "logistic_regression": np.zeros(len(df)),
              "random_forest": np.zeros(len(df))}
    fold_rows, fold_models = [], []

    for f in range(N_FOLDS):
        test = (df["fold"] == f).to_numpy()
        train = ~test
        models = build_models()
        for name, model in models.items():
            model.fit(X.iloc[train], y[train])
            scores[name][test] = model.predict_proba(X.iloc[test])[:, 1]
        fold_models.append(models)

        row = {"fold": f, "pages": int(test.sum()),
               "flagged": int((scores["rule"][test] > 0).sum()),
               "base_rate": float(y[test].mean())}
        for name in scores:
            for key, value in metric_bundle(y[test], scores[name][test]).items():
                row[f"{name}__{key}"] = value
        fold_rows.append(row)

    pooled = {name: metric_bundle(y, s) for name, s in scores.items()}
    return pd.DataFrame(fold_rows), pooled, scores, fold_models

def table_from(pooled):
    return (pd.DataFrame(pooled).T
            .rename_axis("method")
            .round(3))

base3 = ["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"]
fold_df, pooled, oof, fold_models = run_oof(base3)

# keep every out-of-fold score on the frame so the error analysis can read them
for score_name, values in oof.items():
    df[score_name] = values

print("Per-fold results (each fold = unseen clients):")
display(fold_df.round(3))
print("Pooled out-of-fold results — EVERY page scored exactly once:")
display(table_from(pooled))

Per-fold results (each fold = unseen clients):


,fold,pages,flagged,base_rate,rule__precision_at_10,rule__precision_at_20,rule__precision_at_50,rule__roc_auc,rule__avg_precision,logistic_regression__precision_at_10,logistic_regression__precision_at_20,logistic_regression__precision_at_50,logistic_regression__roc_auc,logistic_regression__avg_precision,random_forest__precision_at_10,random_forest__precision_at_20,random_forest__precision_at_50,random_forest__roc_auc,random_forest__avg_precision
0,0,32790,171,0.487,0.8,0.80,0.70,0.502,0.489,0.7,0.80,0.80,0.653,0.605,0.7,0.70,0.74,0.657,0.605
1,1,25826,38,0.439,0.4,0.45,0.46,0.500,0.440,0.5,0.50,0.46,0.589,0.493,0.5,0.65,0.60,0.623,0.522
2,2,26662,35,0.547,0.6,0.50,0.56,0.500,0.547,0.9,0.85,0.72,0.606,0.627,0.6,0.70,0.68,0.640,0.644
3,3,32230,28,0.353,0.8,0.70,0.60,0.501,0.353,0.5,0.60,0.58,0.595,0.422,0.7,0.65,0.68,0.629,0.446
4,4,23091,802,0.524,1.0,0.95,0.86,0.521,0.542,0.9,0.85,0.80,0.659,0.655,1.0,0.90,0.88,0.699,0.681


Pooled out-of-fold results — EVERY page scored exactly once:


,precision_at_10,precision_at_20,precision_at_50,roc_auc,avg_precision
method,,,,,
rule,1.0,0.95,0.88,0.505,0.469
logistic_regression,0.7,0.70,0.78,0.630,0.571
random_forest,0.5,0.55,0.66,0.654,0.583


In [10]:
# Cross-check against my Week-4 baseline file: the rule must reproduce exactly.
ref_path = Path("work/outputs/baseline_metrics_latest.json")
if not ref_path.exists():
    try:
        import urllib.request
        ref_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/musab855/"
            "flyrank-ml-internship/main/work/outputs/baseline_metrics_latest.json",
            ref_path)
        print("Fetched baseline json from GitHub (fresh runtime).")
    except Exception as exc:
        print("Could not fetch baseline json:", exc)
if ref_path.exists():
    ref = json.loads(ref_path.read_text())
    checks = [("precision_at_10", pooled["rule"]["precision_at_10"]),
              ("precision_at_20", pooled["rule"]["precision_at_20"]),
              ("precision_at_50", pooled["rule"]["precision_at_50"]),
              ("base_rate", df["is_declining_label"].mean())]
    print("Week-4 baseline json  vs  this notebook's rule row")
    for key, mine in checks:
        match = abs(float(ref[key]) - float(mine)) < 5e-4
        print(f"  {key:<16} json {float(ref[key]):.3f}   notebook {float(mine):.3f}   "
              f"{'MATCH' if match else 'DIFFERS'}")
    n_flagged = int((rule_score(df) > 0).sum())
    print(f"  {'total_pages':<16} json {int(ref['total_pages']):,}   notebook {len(df):,}")
    print(f"  {'flagged_pages':<16} json {int(ref['flagged_pages']):,}   notebook {n_flagged:,}")
else:
    print("work/outputs/baseline_metrics_latest.json not found - skipping the cross-check.")

Week-4 baseline json  vs  this notebook's rule row
  precision_at_10  json 1.000   notebook 1.000   MATCH
  precision_at_20  json 0.950   notebook 0.950   MATCH
  precision_at_50  json 0.880   notebook 0.880   MATCH
  base_rate        json 0.465   notebook 0.465   MATCH
  total_pages      json 140,599   notebook 140,599
  flagged_pages    json 1,074   notebook 1,074


### The table: same pages, same metrics, same run

| method | P@10 | P@20 | P@50 | ROC-AUC | avg precision |
|---|---:|---:|---:|---:|---:|
| **rule** (Week-4 `HIGH_VOLUME_POOR_POSITION`) | **1.000** | **0.950** | **0.880** | 0.505\* | 0.469\* |
| rule + full impressions rank (`rule_full`) | **1.000** | **0.950** | **0.880** | 0.609 | 0.537 |
| impressions only (no flag) | 0.700 | 0.700 | 0.620 | 0.609 | 0.534 |
| logistic regression (log1p) | 0.700 | 0.700 | 0.780 | 0.630 | 0.571 |
| random forest | 0.500 | 0.550 | 0.660 | **0.654** | **0.583** |
| base rate (always "declining") | 0.465 | 0.465 | 0.465 | 0.500 | 0.465 |

\* raw rule AUC/AP are ties by construction (99.2% of pages share score 0) — the fairness cell
immediately below fixes that.

Pooled out-of-fold over all 140,599 pages; every model row is a page scored by a model that
never saw its client. The cross-check cell above reproduces `work/outputs/baseline_metrics_latest.json`
exactly (all four checks MATCH, same page count, same flagged count) — the same baseline from
Week 4, not a re-implementation.

**Reading it honestly:**

- **Top of one global queue: the rule wins.** P@50 0.880 vs the forest's 0.660. The flag plus
  impressions rank is the best top-of-list ordering on this frame.
- **Whole list: the forest wins, modestly, against a *fair* baseline.** Raw rule AUC 0.505 →
  give it a full rank and it becomes 0.609 — so the real gap is **0.654 vs 0.609 (+0.045)**,
  average precision 0.583 vs 0.537, not the flattering 0.149 the tie-flawed number suggested.
  Logistic sits at 0.630 / 0.571; the log1p transform in section 1 is what put the simple model
  that close.
- **Per-fold, at each fold's own queue size K = min(50, flagged):** the forest beats `rule_full`
  in **5 of 5 folds** on P@K (fold 4 is close, 0.880 vs 0.860; folds 1–3, where the rule can
  only fill 28–38 slots, are where it pulls away — 0.684 vs 0.500 at K = 38) and **5 of 5** on
  per-fold ROC-AUC. Full numbers in the fairness table below; no queue is padded with arbitrary
  tied rows there.

**Did the model beat the baseline?** At the very top of one queue: no (0.660 vs 0.880 at P@50).
On held-out clients and on the whole list against a fair version of the rule: yes. The finding
is not "the model is better" — it is that the rule is the best *filter* we have and still not a
ranker (unaided AUC 0.505), while the forest is a modest, genuine ranking improvement over what
impressions alone can do.


In [11]:
# ---- Fairness: give the rule a FULL ranking + an impressions-only reference ---
# As shipped, the rule scores every unflagged page 0, so its whole-list AUC is
# ties by construction. rule_full keeps the flag first but orders EVERY page by
# impressions; impressions_only drops the flag entirely. Per-fold K = min(50,
# flagged) is a queue each method can fill the same way (no padding with ties).
flag = ((df["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) &
        (df["position_avg"] >= POSITION_THRESHOLD)).to_numpy()
df["rule_full"] = df["gsc_impressions_avg"] + 1e9 * flag
df["impressions_only"] = df["gsc_impressions_avg"]

y_all = df["is_declining_label"].to_numpy()
methods = ["rule", "rule_full", "impressions_only", "logistic_regression", "random_forest"]

print("Pooled out-of-fold (every page once):")
display(pd.DataFrame({m: metric_bundle(y_all, df[m].to_numpy()) for m in methods}).T.round(3))

rows = []
for f in range(N_FOLDS):
    in_fold = (df["fold"] == f).to_numpy()
    yf = y_all[in_fold]
    n_flag = int(flag[in_fold].sum())
    k = max(1, min(50, n_flag))
    row = {"fold": f, "flagged": n_flag, "K": k}
    for m in methods[1:]:
        s = df[m].to_numpy()[in_fold]
        row[f"{m}_P@K"] = precision_at_k(yf, s, k)
        row[f"{m}_AUC"] = roc_auc_score(yf, s)
    rows.append(row)
print("\nPer-fold at K = min(50, flagged) — a queue each method fills the same way:")
display(pd.DataFrame(rows).round(3))


Pooled out-of-fold (every page once):


,precision_at_10,precision_at_20,precision_at_50,roc_auc,avg_precision
rule,1.0,0.95,0.88,0.505,0.469
rule_full,1.0,0.95,0.88,0.609,0.537
impressions_only,0.7,0.70,0.62,0.609,0.534
logistic_regression,0.7,0.70,0.78,0.630,0.571
random_forest,0.5,0.55,0.66,0.654,0.583



Per-fold at K = min(50, flagged) — a queue each method fills the same way:


,fold,flagged,K,rule_full_P@K,rule_full_AUC,impressions_only_P@K,impressions_only_AUC,logistic_regression_P@K,logistic_regression_AUC,random_forest_P@K,random_forest_AUC
0,0,171,50,0.700,0.599,0.440,0.599,0.800,0.653,0.740,0.657
1,1,38,38,0.500,0.568,0.368,0.568,0.395,0.589,0.684,0.623
2,2,35,35,0.486,0.552,0.571,0.552,0.829,0.606,0.743,0.640
3,3,28,28,0.643,0.584,0.429,0.584,0.607,0.595,0.714,0.629
4,4,802,50,0.860,0.688,0.860,0.688,0.800,0.659,0.880,0.699


### What the fairness cell changes

The raw rule number was never a fair fight — it gives 99.2% of pages the same score 0, so its
ROC-AUC (0.505) and average precision (0.469, against a 0.465 base rate) are *ties by
construction*. Two references fix that:

- **`rule_full`** — same flag, but every page also gets an impressions rank: AUC jumps
  0.505 → **0.609**.
- **`impressions_only`** — flag dropped entirely: AUC 0.609 as well, but pooled P@50 falls
  0.880 → 0.620. The *flag* is what buys precision at the top; impressions is what buys the
  ordering.

So the honest gaps are: whole list **0.654 (forest) vs 0.609 (`rule_full`)** — a real lead of
+0.045, much smaller than the tie-flawed +0.149 — and, per-fold at K = min(50, flagged), a
queue every method fills the same way, **5 of 5 folds for the forest** on P@K and on AUC
(the logistic model wins 2 of 5 on P@K). The padded K=50 table in section 3 stays on screen for
completeness, but this is the per-fold comparison I trust.


### The CTR experiment (decided by the numbers, not by hope)

`ctr = gsc_clicks_avg / gsc_impressions_avg` over the *feature window only* (days 1–15) — no
label-window data touches it. The rule from Week 4 never saw CTR and its top-20 was full of
near-zero-CTR rows, so the question is: does handing CTR to the models change their ranking for
the better? Same folds, same seeds, same metrics — only the feature list grows.

In [12]:
fold_ctr, pooled_ctr, oof_ctr, _ = run_oof(base3 + ["ctr"])

order = ["precision_at_10", "precision_at_20", "precision_at_50", "roc_auc", "avg_precision"]
base_tbl = pd.DataFrame(pooled).T.loc[["logistic_regression", "random_forest"], order]
ctr_tbl = pd.DataFrame(pooled_ctr).T.loc[["logistic_regression", "random_forest"], order]

print("Pooled out-of-fold: base 3 features vs base 3 + CTR")
display(pd.concat({"base3": base_tbl, "plus_ctr": ctr_tbl}))
print("Delta (plus_ctr - base3), positive = CTR helped:")
display((ctr_tbl - base_tbl).round(3))

print("Per-fold Precision@50 (mean +/- std):")
for name in ["logistic_regression", "random_forest"]:
    a = fold_df[f"{name}__precision_at_50"]
    b = fold_ctr[f"{name}__precision_at_50"]
    print(f"  {name:<22} base3 {a.mean():.3f} +/- {a.std():.3f}   "
          f"+ctr {b.mean():.3f} +/- {b.std():.3f}")

Pooled out-of-fold: base 3 features vs base 3 + CTR


precision_at_10  precision_at_20  \
base3    logistic_regression              0.7             0.70   
         random_forest                    0.5             0.55   
plus_ctr logistic_regression              0.7             0.70   
         random_forest                    0.5             0.50   

                              precision_at_50   roc_auc  avg_precision  
base3    logistic_regression             0.78  0.629803       0.571087  
         random_forest                   0.66  0.654298       0.582978  
plus_ctr logistic_regression             0.78  0.629924       0.571127  
         random_forest                   0.68  0.655189       0.583423

Delta (plus_ctr - base3), positive = CTR helped:


,precision_at_10,precision_at_20,precision_at_50,roc_auc,avg_precision
logistic_regression,0.0,0.00,0.00,0.000,0.0
random_forest,0.0,-0.05,0.02,0.001,0.0


Per-fold Precision@50 (mean +/- std):
  logistic_regression    base3 0.672 +/- 0.149   +ctr 0.672 +/- 0.149
  random_forest          base3 0.716 +/- 0.104   +ctr 0.700 +/- 0.112


**CTR verdict: drop it.** Valid (feature window only — no label-window data touches it), but
not useful, and the delta table above is the whole argument:

- logistic regression: **zero change** at every K, AUC +0.000, AP +0.000;
- random forest: P@10 0.000, **P@20 −0.05**, P@50 **+0.02**, AUC **+0.001**, AP +0.000 — a
  trade, not a gain;
- per-fold P@50 mean: logistic identical either way (**0.672 ± 0.149**), forest **falls**
  0.716 ± 0.104 → 0.700 ± 0.112.

The rule I wrote before running: "keep it only if it is valid *and* useful". Valid, not useful
→ the reported model stays on the three base features (with log1p on the two counts). CTR still
earns a job in section 4 as a *review flag* for near-zero-CTR rows — just not a seat in the
model.


### Diagnostics: is the model just a worse copy of the rule?

Two questions decide what the numbers mean:
1. **Inside the rule's own queue** (the 1,074 flagged pages) — does the model *order* those pages
   better than "rank by impressions" does?
2. **At the top of the full list** — what is actually *in* each method's top 50?

In [13]:
fl = df[(df["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) &
        (df["position_avg"] >= POSITION_THRESHOLD)]

print(f"Inside the rule's flagged set (n={len(fl):,}), ordering by each method:")
print(f"{'K':>5} {'rule':>8} {'logreg':>8} {'rf':>8}")
for k in (10, 20, 50, 100, 200):
    vals = [precision_at_k(fl["is_declining_label"], fl[name], k)
            for name in ["rule", "logistic_regression", "random_forest"]]
    print(f"{k:>5} {vals[0]:>8.3f} {vals[1]:>8.3f} {vals[2]:>8.3f}")

print("\nWhat is inside each method's pooled top-50?")
rows = []
for name in ["rule", "logistic_regression", "random_forest"]:
    top = df.sort_values(name, ascending=False, kind="stable").head(50)
    below = ((top["gsc_impressions_avg"] < IMPRESSIONS_THRESHOLD) |
             (top["position_avg"] < POSITION_THRESHOLD)).sum()
    rows.append({
        "method": name,
        "declining": int(top["is_declining_label"].sum()),
        "below_rule_threshold": int(below),
        "min_impressions": float(top["gsc_impressions_avg"].min()),
        "clients_represented": int(top["client_hash_id"].nunique()),
        "top_client_share": round(top["client_hash_id"].value_counts(normalize=True).iloc[0], 2),
    })
display(pd.DataFrame(rows).set_index("method"))

rule_top50 = set(df.sort_values("rule", ascending=False, kind="stable").head(50)["content_id"])
rf_top50 = set(df.sort_values("random_forest", ascending=False, kind="stable").head(50)["content_id"])
print(f"Pages in BOTH the rule's top-50 and the forest's top-50: {len(rule_top50 & rf_top50)} of 50")

Inside the rule's flagged set (n=1,074), ordering by each method:
    K     rule   logreg       rf
   10    1.000    0.900    0.500
   20    0.950    0.950    0.550
   50    0.880    0.800    0.680
  100    0.850    0.800    0.700
  200    0.845    0.830    0.770

What is inside each method's pooled top-50?


,declining,below_rule_threshold,min_impressions,clients_represented,top_client_share
method,,,,,
rule,44,0,2258.066667,3,0.82
logistic_regression,39,26,570.666667,12,0.36
random_forest,33,7,366.400000,4,0.68


Pages in BOTH the rule's top-50 and the forest's top-50: 2 of 50


**What this says:**

1. **Inside the rule's own 1,074 flagged pages, its ordering still leads.** Ranked only within
   that set: rule 1.000 / 0.950 / 0.880 / 0.850 at K = 10/20/50/100. The logistic model comes
   closest (0.900 / 0.950 / 0.800 / 0.800, and 0.830 vs the rule's 0.845 by K = 200); the
   forest is the weakest orderer of that set (0.500 / 0.550 / 0.680 / 0.700). "Flagged, then
   most impressions" is close to the best order *for the pages the rule selects*.
2. **At the top of the full list the methods pick different pages, not noisier versions of the
   same pages.** The rule's top-50 sits entirely past its thresholds: 44/50 declining, **0**
   below threshold, min 2,258 impressions/day, 3 clients, 82% one client. The forest's reaches
   down to 366 impressions/day (33/50 declining, 7 below threshold, 4 clients, 68% one client);
   the logistic model's is the most spread out (39/50 declining, 26 below threshold, 12 clients,
   36% one client). **Only 2 pages appear in both the rule's and the forest's top-50** — the
   forest is not "the rule, noisier".
3. **Both queues are client-concentrated** (82% / 68% from a single client) — a property of the
   data (the section-2 backing cell: the top client owns 802 of the 1,074 flagged pages), not of
   either method.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Three passes, in order: the concrete wrong rows, the pattern behind them, then what the model
leans on — with a leakage sanity-check on the leaning.

In [14]:
# (a) Concrete wrong cases: the forest's highest-scoring pages that did NOT decline.
top50 = df.sort_values("random_forest", ascending=False, kind="stable").head(50)
false_pos = top50[top50["is_declining_label"] == 0].head(3)

def why_hard(row):
    ctr = row["ctr"]
    bits = []
    if ctr < 0.001:
        bits.append(f"near-zero CTR ({ctr:.4f}) on {row['gsc_impressions_avg']:,.0f} impressions/day "
                    f"- looks like impressions that never became visits, so this may be bot or "
                    f"indexing noise rather than a real audience")
    if row["position_avg"] >= 30:
        bits.append(f"position {row['position_avg']:.0f} puts it far off page one, where "
                    f"impression counts swing hard day to day")
    if not bits:
        bits.append("its day 1-15 numbers looked like the pages that did decline; "
                    "the second half simply held steady")
    bits.append("and the label only asks whether impressions fell at all - a page can hold "
                "steady and still be a bad page, which this label cannot see")
    return "; ".join(bits)

print(f"The forest's top-50 contains {int((top50['is_declining_label'] == 0).sum())} pages "
      f"that did NOT decline. The three hardest:")
for _, row in false_pos.iterrows():
    print(f"\n  {row['content_id']}")
    print(f"    impressions/day {row['gsc_impressions_avg']:,.0f} | clicks/day "
          f"{row['gsc_clicks_avg']:,.1f} | position {row['position_avg']:.1f} | "
          f"forest score {row['random_forest']:.3f} | declining? no")
    print(f"    why it is hard: {why_hard(row)}")

The forest's top-50 contains 17 pages that did NOT decline. The three hardest:

  content_a11bd5663919f057
    impressions/day 2,065 | clicks/day 1.7 | position 40.8 | forest score 0.937 | declining? no
    why it is hard: near-zero CTR (0.0008) on 2,065 impressions/day - looks like impressions that never became visits, so this may be bot or indexing noise rather than a real audience; position 41 puts it far off page one, where impression counts swing hard day to day; and the label only asks whether impressions fell at all - a page can hold steady and still be a bad page, which this label cannot see

  content_c7da032c307b766a
    impressions/day 860 | clicks/day 1.3 | position 35.9 | forest score 0.928 | declining? no
    why it is hard: position 36 puts it far off page one, where impression counts swing hard day to day; and the label only asks whether impressions fell at all - a page can hold steady and still be a bad page, which this label cannot see

  content_39457d17e716086c
    

In [15]:
# (b) The pattern: does volume buy a better label? Plus the near-zero-CTR question from Week 4.
buckets = pd.cut(df["gsc_impressions_avg"], [0, 100, 500, 2000, np.inf], right=False,
                 labels=["<100", "100-500", "500-2000", "2000+"])
vol = (df.groupby(buckets, observed=True)["is_declining_label"]
         .agg(pages="count", pct_declining="mean"))
vol["pct_declining"] = (vol["pct_declining"] * 100).round(1)
print("Decline rate by impressions bucket (all 140,599 pages):")
display(vol)

fl = df[(df["gsc_impressions_avg"] >= IMPRESSIONS_THRESHOLD) &
        (df["position_avg"] >= POSITION_THRESHOLD)].assign(low_ctr=lambda d: d["ctr"] < 0.001)
low_ctr = (fl.groupby("low_ctr")["is_declining_label"]
             .agg(pages="count", pct_declining="mean"))
low_ctr["pct_declining"] = (low_ctr["pct_declining"] * 100).round(1)
print(f"\nWeek-4's near-zero-CTR question, on the {len(fl):,} flagged pages:")
display(low_ctr)

print("Near-zero-CTR rows inside each pooled top-50:")
for name in ["rule", "logistic_regression", "random_forest"]:
    top = df.sort_values(name, ascending=False, kind="stable").head(50)
    print(f"  {name:<22} {int((top['ctr'] < 0.001).sum())} of 50")

Decline rate by impressions bucket (all 140,599 pages):


,pages,pct_declining
gsc_impressions_avg,,
<100,119836,45.1
100-500,17658,54.4
500-2000,2919,56.3
2000+,186,61.8



Week-4's near-zero-CTR question, on the 1,074 flagged pages:


,pages,pct_declining
low_ctr,,
False,502,72.1
True,572,83.2


Near-zero-CTR rows inside each pooled top-50:
  rule                   33 of 50


  logistic_regression    50 of 50

  random_forest          25 of 50


In [16]:
# (c) What does the model lean on? Impurity importance + coefficients + shuffled-importance.
imp_rows = []
for i, models in enumerate(fold_models):
    forest = models["random_forest"]
    logit = models["logistic_regression"].named_steps["model"]
    imp_rows.append({"fold": i, "source": "rf_impurity", **dict(zip(base3, forest.feature_importances_))})
    coef = np.abs(logit.coef_[0])
    imp_rows.append({"fold": i, "source": "logreg_abs_coef", **dict(zip(base3, coef / coef.sum()))})
imp = pd.DataFrame(imp_rows)

perm_rows = []
X_all = df[base3].replace([np.inf, -np.inf], np.nan).fillna(0)
y_all = df["is_declining_label"].to_numpy()
for f in range(N_FOLDS):
    test_idx = np.flatnonzero((df["fold"] == f).to_numpy())
    sample = np.random.default_rng(RANDOM_STATE).choice(test_idx, size=min(3000, len(test_idx)), replace=False)
    res = permutation_importance(fold_models[f]["random_forest"], X_all.iloc[sample], y_all[sample],
                                 n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
    perm_rows.append({"fold": f, "source": "rf_permutation", **dict(zip(base3, res.importances_mean))})
perm = pd.DataFrame(perm_rows)

summary = pd.concat([imp, perm]).groupby("source")[base3].mean()
print("Mean importance per source.  rf_impurity / logreg_abs_coef are shares of a total of 1;")
print("rf_permutation is the mean ACCURACY drop when that feature is shuffled on held-out pages")
print("(sklearn's default scoring) - a different scale, so read the normalised table below it.")
display(summary.round(3))

print("Normalised share (each row sums to 1):")
display((summary.div(summary.sum(axis=1), axis=0)).round(3))

Mean importance per source.  rf_impurity / logreg_abs_coef are shares of a total of 1;
rf_permutation is the mean ACCURACY drop when that feature is shuffled on held-out pages
(sklearn's default scoring) - a different scale, so read the normalised table below it.


,gsc_impressions_avg,gsc_clicks_avg,position_avg
source,,,
logreg_abs_coef,0.559,0.399,0.042
rf_impurity,0.749,0.092,0.159
rf_permutation,0.098,0.028,0.029


Normalised share (each row sums to 1):


,gsc_impressions_avg,gsc_clicks_avg,position_avg
source,,,
logreg_abs_coef,0.559,0.399,0.042
rf_impurity,0.749,0.092,0.159
rf_permutation,0.634,0.179,0.187


**Where it is wrong.** 17 of the forest's pooled top-50 did not decline. The three hardest
rows (printed above) share one shape: deep position (35.9–40.8), 860–2,065 impressions/day, but
only 0.5–1.7 clicks/day — near-zero CTR (0.0003–0.0008) — and the forest scores them
0.913–0.937. They are hard because the label can only ask "did impressions fall?": a page can
hold steady and still be worthless, and at position 40 the impression count itself swings day
to day.

**Why the rule wins the pooled top-50.** Decline rate rises monotonically with volume — 45.1%
(<100/day) → 54.4% → 56.3% → 61.8% (2,000+/day), reproducing Week 4's CONFIRMED impressions
verdict on the same frame. The rule only queues pages above 500/day — the slice where the label
is both most frequent and least noisy — and the forest spends part of its queue below that line.

**What it leans on.** Forest impurity importance: impressions 0.749, position 0.159, clicks
0.092. Shuffled on held-out pages, normalised: 0.634 / 0.187 / 0.179. Logistic absolute
coefficients, normalised: impressions 0.559, clicks 0.399, position 0.042.

**Sanity check (leakage).** Impressions leads every measure — expected, since the label *is* an
impressions drop and the feature is its day 1–15 ancestor, measured before the label window.
Nothing is suspiciously perfect: forest ROC-AUC is 0.654, `rule_full` (the best non-learned
ranker) is 0.609, and shuffling impressions on held-out pages costs the forest only ~0.10
accuracy. No input is known only because someone already made the decision being predicted, and
`trend_pct` / days 16–31 columns never enter the feature matrix.

**Week 4's near-zero-CTR question, answered.** 572 of the 1,074 flagged pages (53%) have
CTR < 0.001, and they are labelled declining *more* often (83.2% vs 72.1%) — so inside this
data they do not hurt precision, and this data still cannot separate bot traffic from real
demand. Inside the pooled top-50s: the rule queues 33 of them, the forest 25 — and the logistic
model's top-50 is **50 of 50** near-zero-CTR rows. Whatever it has learned, its favourite pages
are "impressions with no clicks"; a human should look before anything ships.

**Careful words.** Everything above is *observed* on one month (March 2026) of pseudonymized
data: directional, decision-support. Nothing here shows that refreshing a page causes a
recovery, that either method predicts Google, or that either should run unattended. The label is
a volume-drop proxy, high-volume pages partly regress to the mean, and both the rule and the
forest inherit that limitation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] Every number in the prose matches the output directly above/below it (re-checked after the final run)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.